# 1. Dense + BM25 단순 병합

In [ ]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 5

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

print("Chroma 연결 완료")

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print("Dense Retriever 생성 완료")

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"BM25용 전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K
print("BM25 Retriever 생성 완료")

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 저장
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    # -------------------------
    # 단순 병합 + 중복 제거
    # -------------------------
    hybrid_results = []
    seen = set()

    for doc in dense_results + bm25_results:
        # 문서 고유 식별용 key
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )

        if doc_key not in seen:
            seen.add(doc_key)
            hybrid_results.append(doc)

        if len(hybrid_results) >= TOP_K:
            break

    comparison_results[query] = {
        "dense": [],
        "bm25": [],
        "hybrid": []
    }

    print("\n" + "=" * 120)
    print(f"질문: {query}")
    print("=" * 120)

    print("\n[DENSE]")
    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[DENSE {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

    print("\n[BM25]")
    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[BM25 {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

    print("\n[HYBRID]")
    for i, doc in enumerate(hybrid_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[HYBRID {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

print("\n저장 완료")
print(f"comparison_results 키 개수: {len(comparison_results)}")
print(f"df_results 행 개수: {len(df_results)}")

In [ ]:
df_results.to_csv("file/csv/hybrid_단순_results.csv", index=False, encoding="utf-8-sig")

In [ ]:
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

dense_docs = comparison_results[query]["dense"]
bm25_docs = comparison_results[query]["bm25"]
hybrid_docs = comparison_results[query]["hybrid"]

dense_set = set(item["page_content"] for item in dense_docs)
bm25_set = set(item["page_content"] for item in bm25_docs)
hybrid_set = set(item["page_content"] for item in hybrid_docs)

bm25_only = bm25_set - dense_set
bm25_added_to_hybrid = bm25_only & hybrid_set

print("BM25에만 있던 문서 수:", len(bm25_only))
print("그 중 Hybrid에 포함된 문서 수:", len(bm25_added_to_hybrid))

for i, doc in enumerate(bm25_added_to_hybrid, 1):
    print(f"\n[BM25 only -> Hybrid 반영 {i}]")
    print(doc[:500])

In [ ]:
query = "주정차 위반 과태료 기준은?"

dense_top1 = comparison_results[query]["dense"][0]["page_content"]
hybrid_docs = [item["page_content"] for item in comparison_results[query]["hybrid"]]

print("Dense 1위 문서가 Hybrid에 남아있나?:", dense_top1 in hybrid_docs)

if dense_top1 in hybrid_docs:
    hybrid_rank = hybrid_docs.index(dense_top1) + 1
    print("Hybrid에서 순위:", hybrid_rank)
else:
    print("Hybrid에서 사라짐")

# 2. RRF

In [ ]:
import chromadb
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 5
RRF_K = 60

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

print("Chroma 연결 완료")

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print("Dense Retriever 생성 완료")

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"BM25용 전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K
print("BM25 Retriever 생성 완료")

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 RRF 융합
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    rrf_scores = defaultdict(float)
    doc_store = {}

    # Dense 순위 반영
    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc

    # BM25 순위 반영
    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc

    # 점수순 정렬
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    hybrid_results = [doc_store[doc_key] for doc_key, _ in sorted_rrf[:TOP_K]]

    comparison_results[query] = {
        "dense": [],
        "bm25": [],
        "hybrid_rrf": []
    }

    print("\n" + "=" * 120)
    print(f"질문: {query}")
    print("=" * 120)

    print("\n[DENSE]")
    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[DENSE {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

    print("\n[BM25]")
    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[BM25 {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

    print("\n[HYBRID_RRF]")
    for i, doc in enumerate(hybrid_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid_rrf"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid_rrf",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[HYBRID_RRF {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

print("\n저장 완료")
print(f"comparison_results 키 개수: {len(comparison_results)}")
print(f"df_results 행 개수: {len(df_results)}")

In [ ]:
df_results.to_csv("file/csv/hybrid_rrf_results.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_results[df_results['query'] == queries[1]]

## 2-2) rrf score check

In [ ]:
import chromadb
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 10
RRF_K = 60

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)


# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )


# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 RRF 융합
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    rrf_scores = defaultdict(float)
    doc_store = {}
    dense_rank_map = {}
    bm25_rank_map = {}

    # Dense 순위 반영
    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        dense_rank_map[doc_key] = rank

    # BM25 순위 반영
    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        bm25_rank_map[doc_key] = rank

    # 점수순 정렬
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    comparison_results[query] = {
        "dense": [],
        "bm25": [],
        "hybrid_rrf": []
    }
    
    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "rrf_score": None,
            "dense_rank": i,
            "bm25_rank": None,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25",
            "rank": i,
            "rrf_score": None,
            "dense_rank": None,
            "bm25_rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    for final_rank, (doc_key, score) in enumerate(sorted_rrf[:TOP_K], start=1):
        doc = doc_store[doc_key]

        item = {
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid_rrf"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid_rrf",
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

print("\n저장 완료")

In [ ]:
df_results.to_csv("file/csv/dense_bm25_rrf_score_compare_topk_10.csv", index=False, encoding="utf-8-sig")

# 2-3) 수동 체크

In [ ]:
df = pd.read_csv("file/csv/dense_bm25_rrf_score_compare_topk_10.csv")

In [ ]:
for i in range(len(queries)):
    df_i = df[df["query"] == queries[i]]
    df_i.to_csv(f"file/csv/df{i+1}.csv", index=False, encoding="utf-8-sig")
    print(f"df{i+1}.csv 저장 완료")

In [ ]:
df_score_man = pd.DataFrame(columns=["질문", "dense", "bm25", "rrf", "결과"])
df_score_man['질문'] = queries

In [ ]:
df_score_man.to_csv("file/csv/df_score_man.csv", index=False, encoding="utf-8-sig")

In [ ]:
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

ls1 = queries[0].split(' ')
ls2 = queries[1].split(' ')
ls3 = queries[2].split(' ')
ls4 = queries[3].split(' ')
ls5 = queries[4].split(' ')

In [ ]:
ls1[2] = ls1[2][0:-1]
ls1.remove(ls1[-1])
ls2.remove(ls2[-1])
ls3.remove(ls3[-1])
ls4.remove(ls4[-1])
ls5.remove(ls5[-1])
ls5.remove(ls5[-1])

In [ ]:
ls5

In [ ]:
for query in queries:
    print("원문:", query)
    print("토큰:", bm25_retriever.preprocess_func(query))
    print("-" * 80)

# 3. Custom 토큰화

In [ ]:
import re
import chromadb
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 10
RRF_K = 60

# =========================
# BM25용 커스텀 전처리
# =========================
def custom_preprocess(text):
    text = re.sub(r"[^\w\s]", " ", text)
    tokens = text.split()

    cleaned_tokens = []

    for token in tokens:
        token = token.strip()
        token = re.sub(
            r"(은|는|이|가|을|를|에|의|도|로|과|와|에서|에게|께|이다|인가|입니까|있다|없다)$",
            "",
            token
        )

        if token:
            cleaned_tokens.append(token)

    return cleaned_tokens

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=custom_preprocess
)
bm25_retriever.k = TOP_K

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# query 토큰 저장
# =========================
query_token_map = {}
for query in queries:
    query_token_map[query] = custom_preprocess(query)

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 RRF 융합
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    rrf_scores = defaultdict(float)
    doc_store = {}
    dense_rank_map = {}
    bm25_rank_map = {}

    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        dense_rank_map[doc_key] = rank

    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        bm25_rank_map[doc_key] = rank

    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    comparison_results[query] = {
        "query_tokens": query_token_map[query],
        "dense": [],
        "bm25": [],
        "hybrid_rrf": []
    }

    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "rrf_score": None,
            "dense_rank": i,
            "bm25_rank": None,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25_custom",
            "rank": i,
            "rrf_score": None,
            "dense_rank": None,
            "bm25_rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    for final_rank, (doc_key, score) in enumerate(sorted_rrf[:TOP_K], start=1):
        doc = doc_store[doc_key]

        item = {
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid_rrf"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid_rrf",
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

# =========================
# CSV 저장
# =========================
df_results.to_csv("file/csv/dense_bm25_custom_rrf_compare.csv", index=False, encoding="utf-8-sig")

## 3-1) 한국어 형태소 custom 토큰화

In [ ]:
import re
import chromadb
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from konlpy.tag import Okt

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 10
RRF_K = 60

# =========================
# 형태소 분석기 준비
# =========================
okt = Okt()

# =========================
# 형태소 기반 BM25 전처리
# =========================
def custom_preprocess(text):
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    pos_tags = okt.pos(text, norm=True, stem=True)

    tokens = []
    for word, pos in pos_tags:
        # 명사 위주 + 숫자/영문 혼합 대응
        if pos in ["Noun", "Alpha", "Number"]:
            if len(word) >= 2:
                tokens.append(word)

    return tokens

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=custom_preprocess
)
bm25_retriever.k = TOP_K

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# query 토큰 저장
# =========================
query_token_map = {}
for query in queries:
    query_token_map[query] = custom_preprocess(query)

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 RRF 융합
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    rrf_scores = defaultdict(float)
    doc_store = {}
    dense_rank_map = {}
    bm25_rank_map = {}

    # Dense 순위 반영
    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        dense_rank_map[doc_key] = rank

    # BM25 순위 반영
    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        bm25_rank_map[doc_key] = rank

    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    comparison_results[query] = {
        "query_tokens": query_token_map[query],
        "dense": [],
        "bm25_morph": [],
        "hybrid_rrf": []
    }

    # Dense 저장
    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "rrf_score": None,
            "dense_rank": i,
            "bm25_rank": None,
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # BM25 저장
    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25_morph"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25_morph",
            "rank": i,
            "rrf_score": None,
            "dense_rank": None,
            "bm25_rank": i,
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # Hybrid RRF 저장
    for final_rank, (doc_key, score) in enumerate(sorted_rrf[:TOP_K], start=1):
        doc = doc_store[doc_key]

        item = {
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid_rrf"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid_rrf",
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

# =========================
# 전체 CSV 저장
# =========================
df_results.to_csv("file/csv/dense_bm25_morph_rrf_compare.csv", index=False, encoding="utf-8-sig")

# =========================
# 질문별 CSV 저장
# =========================
# for i, query in enumerate(queries, start=1):
#     df_q = df_results[df_results["query"] == query]
#     df_q.to_csv(f"morph_q{i}.csv", index=False, encoding="utf-8-sig")

## 3-2) 형태소 분석 + 복합명사 보정 + stopword 반영

In [ ]:
import re
import chromadb
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from konlpy.tag import Okt

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 10
RRF_K = 60

# =========================
# 형태소 분석기
# =========================
okt = Okt()

# =========================
# stopword 후보
# =========================
STOPWORDS = {
    "기준", "여부", "가능", "얼마", "등", "통해", "경우", "사항"
}

# =========================
# 복합명사 후보 사전
# query/doc 둘 다에서 같은 방식으로 보정
# =========================
COMPOUND_TERMS = [
    "어린이보호구역",
    "어린이 보호구역",
    "보호구역",
    "제한속도",
    "속도제한",
    "최고속도",
    "통행속도",
    "스쿨존",
    "속도위반",
    "주정차위반",
    "주정차 위반",
    "주차위반",
    "정차위반",
    "과태료",
    "범칙금",
    "부과기준",
    "면허취소",
    "면허정지",
    "운전면허",
    "음주운전",
    "취소처분",
    "주차단속",
    "불법주차",
    "신고",
    "견인",
    "조치",
]

# 공백 제거 버전도 같이 관리
COMPOUND_TERMS_NORMALIZED = {term.replace(" ", ""): term.replace(" ", "") for term in COMPOUND_TERMS}


# =========================
# 전처리 함수
# =========================
def custom_preprocess(text: str):
    # 1) 기초 정리
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 2) 형태소 분석
    pos_tags = okt.pos(text, norm=True, stem=True)

    base_tokens = []
    for word, pos in pos_tags:
        if pos in ["Noun", "Alpha", "Number"]:
            if len(word) >= 2:
                base_tokens.append(word)

    # 3) 원문 공백 제거 문자열 준비
    compact_text = re.sub(r"\s+", "", text)

    # 4) 복합명사 보정
    #    원문에 실제로 존재하는 복합명사를 추가
    compound_tokens = []
    for compact_term in COMPOUND_TERMS_NORMALIZED:
        if compact_term in compact_text:
            compound_tokens.append(compact_term)

    # 5) 합치기
    tokens = base_tokens + compound_tokens

    # 6) stopword 제거 + 중복 제거(순서 유지)
    cleaned = []
    seen = set()

    for token in tokens:
        token = token.strip()
        if not token:
            continue
        if token in STOPWORDS:
            continue
        if len(token) < 2:
            continue
        if token not in seen:
            seen.add(token)
            cleaned.append(token)

    return cleaned


# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()


# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# Dense Retriever
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

# =========================
# BM25 Retriever
# =========================
bm25_retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=custom_preprocess
)
bm25_retriever.k = TOP_K

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# query token 저장
# =========================
query_token_map = {}
for query in queries:
    query_token_map[query] = custom_preprocess(query)

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 RRF
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    rrf_scores = defaultdict(float)
    doc_store = {}
    dense_rank_map = {}
    bm25_rank_map = {}

    # Dense 순위 반영
    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        dense_rank_map[doc_key] = rank

    # BM25 순위 반영
    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        bm25_rank_map[doc_key] = rank

    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    comparison_results[query] = {
        "query_tokens": query_token_map[query],
        "dense": [],
        "bm25_compound": [],
        "hybrid_rrf": []
    }

    # Dense 저장
    for i, doc in enumerate(dense_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "rrf_score": None,
            "dense_rank": i,
            "bm25_rank": None,
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # BM25 저장
    for i, doc in enumerate(bm25_results, 1):
        item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25_compound"].append(item)

        rows.append({
            "query": query,
            "retriever": "bm25_compound",
            "rank": i,
            "rrf_score": None,
            "dense_rank": None,
            "bm25_rank": i,
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # Hybrid RRF 저장
    for final_rank, (doc_key, score) in enumerate(sorted_rrf[:TOP_K], start=1):
        doc = doc_store[doc_key]

        item = {
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["hybrid_rrf"].append(item)

        rows.append({
            "query": query,
            "retriever": "hybrid_rrf",
            "rank": final_rank,
            "rrf_score": score,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
            "query_tokens": ", ".join(query_token_map[query]),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

# =========================
# CSV 저장
# =========================
df_results.to_csv("file/csv/dense_bm25_compound_rrf_compare.csv", index=False, encoding="utf-8-sig")

# 4. query expansion

In [ ]:
import re
import chromadb
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 10

# =========================
# BM25용 커스텀 전처리
# =========================
def custom_preprocess(text):
    text = re.sub(r"[^\w\s]", " ", text)
    tokens = text.split()

    cleaned_tokens = []

    for token in tokens:
        token = token.strip()
        token = re.sub(
            r"(은|는|이|가|을|를|에|의|도|로|과|와|에서|에게|께|이다|인가|입니까|있다|없다)$",
            "",
            token
        )
        if token:
            cleaned_tokens.append(token)

    return cleaned_tokens

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=custom_preprocess
)
bm25_retriever.k = TOP_K

# =========================
# 원문 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 실험용 query expansion
# =========================
query_expansion_map = {
    "어린이 보호구역 제한속도는 얼마인가?": "어린이 보호구역 제한속도 속도제한 어린이보호구역 통행속도 최고속도",
    "스쿨존 속도위반 기준은?": "스쿨존 어린이 보호구역 어린이보호구역 속도위반 제한속도 속도제한 최고속도",
    "주정차 위반 과태료 기준은?": "주정차 위반 주차위반 정차위반 과태료 부과기준 범칙금 주정차위반",
    "음주운전 면허취소 기준은?": "음주운전 면허취소 운전면허 취소 운전면허의 취소 정지 취소처분 취소기준",
    "불법주차 신고 가능 여부는?": "불법주차 주차위반 주정차위반 신고 조치 견인 과태료 단속 주차위반에 대한 조치"
}

# =========================
# 결과 저장용
# =========================
doc_rows = []
token_rows = []

# =========================
# 검색 실행
# =========================
for query in queries:
    expanded_query = query_expansion_map[query]

    original_tokens = custom_preprocess(query)
    expanded_tokens = custom_preprocess(expanded_query)

    dense_results = dense_retriever.invoke(query)
    bm25_original_results = bm25_retriever.invoke(query)
    bm25_expanded_results = bm25_retriever.invoke(expanded_query)

    # -------------------------
    # Dense 결과 저장
    # -------------------------
    for i, doc in enumerate(dense_results, start=1):
        doc_rows.append({
            "query": query,
            "query_type": "original",
            "used_query": query,
            "retriever": "dense",
            "rank": i,
            "query_tokens": None,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # -------------------------
    # BM25 original 결과 저장 + 토큰 count 저장
    # -------------------------
    for i, doc in enumerate(bm25_original_results, start=1):
        doc_rows.append({
            "query": query,
            "query_type": "original",
            "used_query": query,
            "retriever": "bm25_original",
            "rank": i,
            "query_tokens": ", ".join(original_tokens),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        doc_tokens = custom_preprocess(doc.page_content)
        token_counter = Counter(doc_tokens)

        for token in original_tokens:
            token_rows.append({
                "query": query,
                "query_type": "original",
                "used_query": query,
                "retriever": "bm25_original",
                "doc_rank": i,
                "query_token": token,
                "count_in_page_content": token_counter[token],
                "metadata": str(doc.metadata),
                "page_content_preview": doc.page_content[:300]
            })

    # -------------------------
    # BM25 expanded 결과 저장 + 토큰 count 저장
    # -------------------------
    for i, doc in enumerate(bm25_expanded_results, start=1):
        doc_rows.append({
            "query": query,
            "query_type": "expanded",
            "used_query": expanded_query,
            "retriever": "bm25_expanded",
            "rank": i,
            "query_tokens": ", ".join(expanded_tokens),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        doc_tokens = custom_preprocess(doc.page_content)
        token_counter = Counter(doc_tokens)

        for token in expanded_tokens:
            token_rows.append({
                "query": query,
                "query_type": "expanded",
                "used_query": expanded_query,
                "retriever": "bm25_expanded",
                "doc_rank": i,
                "query_token": token,
                "count_in_page_content": token_counter[token],
                "metadata": str(doc.metadata),
                "page_content_preview": doc.page_content[:300]
            })

# =========================
# DataFrame 생성
# =========================
df_doc_results = pd.DataFrame(doc_rows)
df_token_counts = pd.DataFrame(token_rows)

# =========================
# CSV 저장
# =========================
df_doc_results.to_csv("file/csv/bm25_query_expansion_doc_results.csv", index=False, encoding="utf-8-sig")
df_token_counts.to_csv("file/csv/bm25_query_expansion_token_counts.csv", index=False, encoding="utf-8-sig")

# =========================
# 질문별 CSV 저장
# =========================
# for i, query in enumerate(queries, start=1):
#     df_doc_q = df_doc_results[df_doc_results["query"] == query]
#     df_tok_q = df_token_counts[df_token_counts["query"] == query]

#     df_doc_q.to_csv(f"query_expansion_doc_q{i}.csv", index=False, encoding="utf-8-sig")
#     df_tok_q.to_csv(f"query_expansion_token_q{i}.csv", index=False, encoding="utf-8-sig")